In [ ]:
%%configure -f
{
    "defaultLakehouse": { "name": "GoldLakehouse" }
}

# Gold - Business Marts (Materialized Lake Views)

Builds the **Gold layer** as **Materialized Lake Views (MLVs)** in `GoldLakehouse`.

**Why MLVs:** each view is a *physically stored, pre-computed Delta table* whose logic is
declared once and refreshed by Fabric on a schedule (in dependency order, with lineage and
retries). Downstream consumers - Power BI Direct Lake, the SQL endpoint, notebooks, ML - all
read the same governed result instead of re-evaluating heavy joins per query.

**Sources:** MLVs can only read tables **within their own lakehouse**, so the Silver `dbo`
tables are surfaced here as the **`silver`** schema via OneLake shortcuts (zero-copy). The
marts are written to the **`dbo`** schema of `GoldLakehouse`.

**Two tiers (the DAG Fabric refreshes automatically):**

| Tier | View | Grain | Feeds |
| --- | --- | --- | --- |
| 1 | `dbo.student_term_fact` | student x term | Q1, Q2, `student_profile` |
| 1 | `dbo.section_economics` | course section | Q3 |
| 1 | `dbo.review_enriched` | AI review | Q4, Q5 |
| 1 | `dbo.student_profile` | student x exam | Q6 |
| 2 | `dbo.q1_retention_by_outcome` | outcome | Retention signals |
| 2 | `dbo.q2_graduation_progress` | status | Credits-to-degree |
| 2 | `dbo.q3_section_cost_effectiveness` | section | Cost per earned credit |
| 2 | `dbo.q4_sentiment_by_grade_band` | grade band | Sentiment vs grade |
| 2 | `dbo.q5_instructor_sentiment` | professor | Instructor quality |
| 2 | `dbo.q6_student_gradexam_bridge` | student x exam | Undergrad -> grad-exam |

> **Refresh is NOT orchestrated from this notebook.** Run it once to author the definitions,
> then schedule refresh from the lakehouse: **GoldLakehouse -> Materialized lake views ->
> Manage -> add a schedule.** Fabric refreshes the views in dependency order.

## Tier 1 - foundational wide marts
The expensive joins happen here, once. Tier 2 answers are cheap aggregations on top of these.

In [ ]:
# Tier 1: one row per student per term. term_recency_rank = 1 marks the latest term.
spark.sql("""
CREATE MATERIALIZED LAKE VIEW IF NOT EXISTS dbo.student_term_fact
COMMENT "One row per student per term: engagement + credit progress (retention & graduation backbone)"
AS
SELECT
    sts.student_id,
    sts.term_id,
    s.current_status,
    s.first_gen,
    s.financial_aid_tier,
    s.program_id,
    pr.program_name,
    pr.required_credits,
    sts.status                                                                AS term_status,
    sts.term_gpa,
    sts.cum_gpa,
    sts.credits_earned,
    sts.cum_credits_earned,
    sts.attendance_pct,
    sts.lms_logins,
    sts.advising_visits,
    ROUND(100.0 * sts.cum_credits_earned / NULLIF(pr.required_credits, 0), 1) AS pct_to_degree,
    ROW_NUMBER() OVER (PARTITION BY sts.student_id ORDER BY sts.term_id DESC)  AS term_recency_rank
FROM silver.student_term_status sts
JOIN silver.student  s  ON s.student_id  = sts.student_id
JOIN silver.program  pr ON pr.program_id = s.program_id
""")
print("Created dbo.student_term_fact")

In [ ]:
# Tier 1: per-section economics. credits_earned uses the actual course credit-hours
# (grade_points >= 1.0 counts as a completed course).
spark.sql("""
CREATE MATERIALIZED LAKE VIEW IF NOT EXISTS dbo.section_economics
COMMENT "Per-section cost and completed-credit economics"
AS
SELECT
    cs.section_id,
    c.crs_id,
    c.crs_nmbr,
    c.crs_name,
    t.term_id,
    t.term_name,
    p.prof_id,
    p.prof_name,
    cs.seats_filled,
    cs.capacity,
    cs.instructor_cost + cs.room_cost                                       AS section_cost,
    SUM(CASE WHEN e.grade_points >= 1.0 THEN c.credits ELSE 0 END)          AS credits_earned,
    ROUND((cs.instructor_cost + cs.room_cost)
          / NULLIF(SUM(CASE WHEN e.grade_points >= 1.0 THEN c.credits ELSE 0 END), 0), 2)
                                                                            AS cost_per_earned_credit
FROM silver.course_section cs
JOIN silver.course     c ON c.crs_id  = cs.crs_id
JOIN silver.term       t ON t.term_id = cs.term_id
JOIN silver.professor  p ON p.prof_id = cs.prof_id
JOIN silver.enrollment e ON e.section_id = cs.section_id
GROUP BY cs.section_id, c.crs_id, c.crs_nmbr, c.crs_name, t.term_id, t.term_name,
         p.prof_id, p.prof_name, cs.seats_filled, cs.capacity, cs.instructor_cost, cs.room_cost
""")
print("Created dbo.section_economics")

In [ ]:
# Tier 1: each AI review enriched with the grade earned and the instructor.
# LEFT JOINs keep every review (so Q4 counts are complete) even when the instructor is unknown;
# Q5 filters to rows where the professor resolved. Data-quality gate: sentiment must be in [-1, 1].
spark.sql("""
CREATE MATERIALIZED LAKE VIEW IF NOT EXISTS dbo.review_enriched
(
    CONSTRAINT valid_sentiment CHECK (sentiment_score BETWEEN -1 AND 1) ON MISMATCH DROP
)
COMMENT "AI course-review signal enriched with grade band and instructor"
AS
SELECT
    r.student_id,
    r.course_id,
    r.sentiment_score,
    r.would_recommend,
    r.quality_points,
    CASE WHEN r.quality_points >= 3.0 THEN 'A/B (>=3.0)'
         WHEN r.quality_points >= 2.0 THEN 'C (2.0-2.9)'
         ELSE 'D/F (<2.0)' END AS grade_band,
    e.section_id,
    cs.prof_id,
    p.prof_name,
    d.dept_code,
    c.crs_nmbr
FROM silver.course_review_signal_ai r
LEFT JOIN silver.enrollment     e  ON e.student_id = r.student_id AND e.crs_id = r.course_id AND e.attempt_number = 1
LEFT JOIN silver.course_section cs ON cs.section_id = e.section_id
LEFT JOIN silver.professor      p  ON p.prof_id = cs.prof_id
LEFT JOIN silver.department     d  ON d.dept_id = p.dept_id
LEFT JOIN silver.course         c  ON c.crs_id = r.course_id
""")
print("Created dbo.review_enriched")

In [ ]:
# Tier 1: per-student profile joined across the SQL star and the Cosmos grad-exam outcome.
# Depends on the Tier-1 dbo.student_term_fact MLV (MLV-on-MLV, inside GoldLakehouse).
spark.sql("""
CREATE MATERIALIZED LAKE VIEW IF NOT EXISTS dbo.student_profile
COMMENT "Per-student profile: latest cumulative progress + grad-exam outcome (SQL x Cosmos)"
AS
SELECT
    s.student_id,
    s.student_name,
    s.current_status,
    pr.program_name,
    stf.cum_gpa,
    stf.cum_credits_earned,
    stf.pct_to_degree,
    g.exam,
    g.score,
    g.admit_threshold,
    g.passed,
    ROUND(g.relevant_gpa, 2) AS relevant_gpa
FROM silver.student s
JOIN      silver.program          pr  ON pr.program_id = s.program_id
LEFT JOIN dbo.student_term_fact    stf ON stf.student_id = s.student_id AND stf.term_recency_rank = 1
LEFT JOIN silver.grad_exam_result  g   ON g.student_id = s.student_id AND g.attempt_number = 1
""")
print("Created dbo.student_profile")

## Tier 2 - business-question answers
Thin aggregations on the Tier-1 marts. Each answers one demo question.

In [ ]:
# Q1 - Retention: which term-1 signals separate dropouts from the retained?
spark.sql("""
CREATE MATERIALIZED LAKE VIEW IF NOT EXISTS dbo.q1_retention_by_outcome
COMMENT "Q1 Retention: term-1 signals by outcome"
AS
SELECT
    CASE WHEN current_status = 'Withdrawn' THEN 'Dropped out'
         ELSE 'Retained / Graduated' END        AS outcome,
    COUNT(*)                                     AS students,
    ROUND(AVG(term_gpa), 2)                      AS avg_term1_gpa,
    ROUND(AVG(attendance_pct), 0)                AS avg_attendance,
    ROUND(AVG(lms_logins), 0)                    AS avg_lms_logins,
    ROUND(AVG(financial_aid_tier), 2)            AS avg_aid_tier,
    ROUND(AVG(CAST(first_gen AS DOUBLE)), 2)     AS pct_first_gen
FROM dbo.student_term_fact
WHERE term_id = 1
GROUP BY 1
""")
print("Created dbo.q1_retention_by_outcome")

In [ ]:
# Q2 - Graduation: latest credit progress vs the program requirement.
spark.sql("""
CREATE MATERIALIZED LAKE VIEW IF NOT EXISTS dbo.q2_graduation_progress
COMMENT "Q2 Graduation: credits-to-degree by current status"
AS
SELECT
    current_status,
    COUNT(*)                        AS students,
    ROUND(AVG(cum_credits_earned), 0) AS avg_credits_earned,
    MIN(required_credits)           AS required_credits,
    ROUND(AVG(pct_to_degree), 1)    AS pct_to_degree
FROM dbo.student_term_fact
WHERE term_recency_rank = 1
GROUP BY current_status
""")
print("Created dbo.q2_graduation_progress")

In [ ]:
# Q3 - Cost-effectiveness: cost per earned credit per section (rank in the report / semantic model).
spark.sql("""
CREATE MATERIALIZED LAKE VIEW IF NOT EXISTS dbo.q3_section_cost_effectiveness
COMMENT "Q3 Cost-effectiveness: cost per earned credit by section"
AS
SELECT
    crs_nmbr,
    term_name,
    prof_name,
    seats_filled,
    capacity,
    section_cost,
    credits_earned,
    cost_per_earned_credit
FROM dbo.section_economics
""")
print("Created dbo.q3_section_cost_effectiveness")

In [ ]:
# Q4 - Reviews: does AI-derived sentiment track the grade earned?
spark.sql("""
CREATE MATERIALIZED LAKE VIEW IF NOT EXISTS dbo.q4_sentiment_by_grade_band
COMMENT "Q4 Reviews: AI sentiment vs grade band"
AS
SELECT
    grade_band,
    COUNT(*)                                       AS reviews,
    ROUND(AVG(sentiment_score), 2)                 AS avg_ai_sentiment,
    ROUND(AVG(CAST(would_recommend AS DOUBLE)), 2) AS pct_recommend
FROM dbo.review_enriched
GROUP BY grade_band
""")
print("Created dbo.q4_sentiment_by_grade_band")

In [ ]:
# Q5 - Instructor quality: professors ranked by AI review sentiment.
spark.sql("""
CREATE MATERIALIZED LAKE VIEW IF NOT EXISTS dbo.q5_instructor_sentiment
COMMENT "Q5 Instructor quality: professors by AI review sentiment"
AS
SELECT
    prof_name,
    dept_code,
    COUNT(*)                       AS reviews,
    ROUND(AVG(sentiment_score), 2) AS avg_ai_sentiment
FROM dbo.review_enriched
WHERE prof_name IS NOT NULL
GROUP BY prof_name, dept_code
""")
print("Created dbo.q5_instructor_sentiment")

In [ ]:
# Q6 - Cross-store: SQL undergrad profile joined to Cosmos grad-exam outcomes on student_id.
spark.sql("""
CREATE MATERIALIZED LAKE VIEW IF NOT EXISTS dbo.q6_student_gradexam_bridge
COMMENT "Q6 Cross-store: undergrad performance vs grad-exam outcome"
AS
SELECT
    student_id,
    student_name,
    program_name,
    current_status,
    cum_gpa,
    cum_credits_earned,
    exam,
    score,
    admit_threshold,
    passed,
    relevant_gpa
FROM dbo.student_profile
WHERE exam IS NOT NULL
""")
print("Created dbo.q6_student_gradexam_bridge")

In [ ]:
# List the materialized lake views and preview the six answers.
display(spark.sql("SHOW MATERIALIZED LAKE VIEWS IN dbo"))

In [ ]:
for q in ["q1_retention_by_outcome", "q2_graduation_progress", "q4_sentiment_by_grade_band"]:
    print("===", q, "===")
    display(spark.table(f"dbo.{q}"))
print("=== q3_section_cost_effectiveness (top 10 worst) ===")
display(spark.sql("SELECT * FROM dbo.q3_section_cost_effectiveness ORDER BY cost_per_earned_credit DESC LIMIT 10"))
print("=== q5_instructor_sentiment (lowest first) ===")
display(spark.sql("SELECT * FROM dbo.q5_instructor_sentiment ORDER BY avg_ai_sentiment ASC"))
print("=== q6_student_gradexam_bridge (sample) ===")
display(spark.sql("SELECT * FROM dbo.q6_student_gradexam_bridge ORDER BY student_id LIMIT 15"))

## Scheduling the refresh (do this once, in the lakehouse UI)

1. Open **GoldLakehouse** -> ribbon **Materialized lake views** -> **Manage**.
2. Review the **lineage** graph - Fabric infers the Tier-1 -> Tier-2 dependency order.
3. **Add a schedule** (e.g. daily) to refresh all views; Fabric runs them in dependency order,
   skips views whose sources did not change, and retries transient failures.

Point a **Direct Lake semantic model** at the Tier-1 marts (`student_term_fact`,
`review_enriched`, `section_economics`, `student_profile`) for interactive slicing, and use the
Tier-2 `q*` marts for instant KPI cards.